# Fluorescence Experiments Overview
``find_fluorescence_timelapse.py`` is a script designed to analyze fluorescence timelapse datasets using OpenAI's language models. The script processes datasets, extracts relevant information from log files, and classifies the data based on predefined criteria. 

This notebook is to analyse the diversity of datasets in the experiment IDs found.

In [1]:
import sys
sys.path.insert(0, "/home/ianyang/alibylite/src") # this script should be run in the alibylite environment, so we can import from src
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path
import openai
from dotenv import load_dotenv
from omero.gateway import BlitzGateway
# Load .env from the script's directory
load_dotenv(".env")

RESULTS_CSV_PATH = "results.csv"
# Load the results CSV
results_df = pd.read_csv(RESULTS_CSV_PATH)

2026-05-14 13:55:25,463 DEBUG [              omero.util.TempFileManager] (MainThread) Added file /home/ianyang/omero/tmp/.lock_tests5rwtfvi.tmp
2026-05-14 13:55:25,463 DEBUG [              omero.util.TempFileManager] (MainThread) Chose global tmpdir: /home/ianyang/omero/tmp
2026-05-14 13:55:25,464 DEBUG [              omero.util.TempFileManager] (MainThread) Using temp dir: /home/ianyang/omero/tmp/omero_ianyang/37138


In [2]:
mask = (results_df["classification"] == "YES") & (pd.to_numeric(results_df["timepoints"], errors="coerce") != 1)
results_df[mask]

,dataset_id,dataset_name,has_log,classification,reason,channels,timepoints,raw_llm_response
0,2493,Ramp_0d5to0pGlc_5h_Msn2Dot6Mig1_EarlyStart_00,True,YES,The experiment has more than one timepoint and...,"GFP_Z, cy5, mCherry_Z",192,CLASSIFICATION: YES \nREASON: The experiment ...
10,577,pHCalibrate6_7_00,True,YES,The experiment has multiple timepoints and use...,"pHluorin405,GFPFast",4,CLASSIFICATION: YES \nREASON: The experiment ...
11,583,pH_med_to_high_00,True,YES,The experiment includes multiple timepoints an...,"pHluorin405,GFPFast,cy5",180,CLASSIFICATION: YES \nREASON: The experiment ...
12,581,pH_med_to_low_00,True,YES,The experiment has multiple timepoints (180 fr...,"pHluorin405,GFPFast,cy5",180,CLASSIFICATION: YES \nREASON: The experiment ...
16,589,pH_med_to_low_00,True,YES,The experiment includes multiple timepoints (f...,"pHluorin405, GFPFast, cy5",180,CLASSIFICATION: YES \nREASON: The experiment ...
...,...,...,...,...,...,...,...,...
1787,4109,TFScreen_plate1_lowN2_00,True,YES,The experiment has multiple timepoints and inc...,"GFP_Z,cy5",24,CLASSIFICATION: YES \nREASON: The experiment ...
1788,4110,TFScreen_plate1_2_lowN2_00,True,YES,The experiment has multiple timepoints (24 fra...,"GFP_Z, cy5",24,CLASSIFICATION: YES \nREASON: The experiment ...
1789,4151,Switchearly_2to0pGlc_Msn2_dIra12_00,True,YES,The experiment involves multiple timepoints an...,"GFP,mCherry,cy5",3,CLASSIFICATION: YES \nREASON: The experiment ...
1792,4203,Aggregates_recovery_00,True,YES,The experiment includes multiple timepoints an...,"GFP_Z,cy5",180,CLASSIFICATION: YES \nREASON: The experiment ...


In [3]:
"""
EXP-26-IY026: Identify fluorescence time-lapse experiments on the lab OMERO server.

Connects to staffa.bio.ed.ac.uk, enumerates every accessible dataset, reads any
attached log file annotation, and asks an LLM to classify whether the experiment
is a fluorescence time-lapse microscopy experiment.

Results are written to results.csv in the same directory.

Usage:
    /home/ianyang/micromamba/envs/alibylite/bin/python3 find_fluorescence_timelapse.py

Requirements:
    - OPENAI_API_KEY env var set
    - omero-py and openai packages installed in the active environment
"""

import csv
import logging
import os
import sys
sys.path.insert(0, "/home/ianyang/alibylite/src") # this script should be run in the alibylite environment, so we can import from src
import time
from pathlib import Path

import openai
from dotenv import load_dotenv
from omero.gateway import BlitzGateway

# Load .env from the script's directory
load_dotenv(".env")

# ---------------------------------------------------------------------------
# Configuration — edit these before running
# ---------------------------------------------------------------------------
SERVER = "staffa.bio.ed.ac.uk"
USERNAME = "upload"
PASSWORD = "gothamc1ty"

# OpenAI API key; falls back to the OPENAI_API_KEY environment variable
API_KEY = os.environ.get("OPENAI_API_KEY")

# OpenAI model used for classification
MODEL = "gpt-5.4"

# Set to True to skip datasets already present in results.csv (useful for resuming a crashed run)
RESUME = False

# Process at most this many datasets; set to None to process all
LIMIT = None

OUTPUT_PATH ="results.csv"
LOG_PATH = "find_fluorescence_timelapse.log"

# Max characters of log text to send to the LLM (keeps tokens low)
LOG_EXCERPT_CHARS = 3000


# ---------------------------------------------------------------------------
# OMERO helpers
# ---------------------------------------------------------------------------

def connect_omero(host: str, username: str, password: str) -> BlitzGateway:
    conn = BlitzGateway(host=host, username=username, passwd=password)
    if not conn.connect():
        raise ConnectionError(f"Could not connect to OMERO at {host}")
    conn.c.enableKeepAlive(60)  # ping the server every 60 s so long runs don't get disconnected
    return conn


def get_all_datasets(conn: BlitzGateway) -> list[tuple[int, str]]:
    """Return (id, name) for every dataset visible to the connected user."""
    return [(int(ds.getId()), ds.getName()) for ds in conn.getObjects("Dataset")]


_MICROSCOPY_LOG_HEADER = "Swain Lab microscope experiment log file"
_MICROSCOPY_LOG_KEYWORDS = ("Experiment details", "Acquisition settings", "Microscope name")


def _is_microscopy_log(text: str) -> bool:
    return _MICROSCOPY_LOG_HEADER in text or any(k in text for k in _MICROSCOPY_LOG_KEYWORDS)


def read_log_annotation(dataset_obj) -> str | None:
    """
    Return the decoded text of the microscopy log file annotation attached to
    the dataset, or None if no such annotation exists.

    Selection priority:
    1. Any .log/.txt file whose content looks like a microscopy log (contains
       the Swain Lab header or "Experiment details").
    2. Any file whose name ends with 'log.txt' (older multiDGUI naming).
    3. Any file ending with '.log' (newer Batgirl format).
    4. First .txt file found (last resort).
    """
    candidates: list[tuple[int, str]] = []  # (priority, text)
    for ann in dataset_obj.listAnnotations():
        if not hasattr(ann, "getFileName"):
            continue  # skip non-file annotations (tags, comments, etc.)
        fname = (ann.getFileName() or "").lower()
        if not (fname.endswith(".log") or fname.endswith(".txt")):
            continue  # only consider plain-text / log files
        raw = b"".join(ann.getFileInChunks())  # stream the file in chunks to avoid loading it all at once
        text = raw.decode("utf-8", errors="ignore")
        if _is_microscopy_log(text):
            return text  # content positively identified — no need to check other annotations
        # File doesn't look like a microscopy log by content; rank it by filename pattern
        # so we can fall back to the most likely candidate if nothing better is found
        if fname.endswith("log.txt"):
            candidates.append((1, text))   # older multiDGUI naming convention
        elif fname.endswith(".log"):
            candidates.append((2, text))   # newer Batgirl format
        else:
            candidates.append((3, text))   # last resort: any .txt file
    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][1]  # return the highest-priority candidate
    return None


# ---------------------------------------------------------------------------
# LLM classification
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """\
You are a lab data curator analysing microscopy experiment metadata from the
Swain lab (yeast live-cell imaging). You receive a dataset name and the first
~3000 characters of a log file (which may be in a newer Swainlab format or an
older multiDGUI format) and must determine whether the experiment is a
fluorescence time-lapse microscopy experiment.

Criteria to check:
1. Time-lapse: more than one timepoint was acquired (look for "frames:",
   "ntimepoints:", or "Number of timepoints" > 1).
2. Fluorescence: at least one fluorescence channel was used. Fluorescence
   channels include: GFP, GFP_Z, GFPFast, mCherry, Citrine, Flavin, mKO2,
   Cy5, cy5, pHluorin405, pHluorin488, NADH, mTurquoise2, tdTomatoFRET.
   Brightfield alone is NOT fluorescence.

Respond ONLY in this exact format (four lines, no extra text):
CLASSIFICATION: YES or NO
REASON: one sentence
CHANNELS: comma-separated channel names found, or UNKNOWN
TIMEPOINTS: integer number of timepoints, or UNKNOWN
"""

USER_TEMPLATE = """\
Dataset ID: {dataset_id}
Dataset name: {dataset_name}

Log file excerpt:
---
{log_excerpt}
---
"""


def classify(
    client: openai.OpenAI,
    dataset_id: int,
    dataset_name: str,
    log_text: str | None,
    model: str = "gpt-5.4",
) -> dict:
    excerpt = log_text[:LOG_EXCERPT_CHARS] if log_text else "(no log file attached)"
    user_msg = USER_TEMPLATE.format(
        dataset_id=dataset_id,
        dataset_name=dataset_name,
        log_excerpt=excerpt,
    )
    response = client.chat.completions.create(
        model=model,
        max_tokens=200,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    )
    text = response.choices[0].message.content.strip()

    result = {
        "dataset_id": dataset_id,
        "dataset_name": dataset_name,
        "has_log": log_text is not None,
        "classification": "UNKNOWN",
        "reason": "",
        "channels": "",
        "timepoints": "", # TODO: determine this from the log text
        # TODO: any other fields? 
        "raw_llm_response": text,
    }
    # Parse the four structured lines from the LLM response
    for line in text.splitlines():
        if line.startswith("CLASSIFICATION:"):
            result["classification"] = line.split(":", 1)[1].strip()
        elif line.startswith("REASON:"):
            result["reason"] = line.split(":", 1)[1].strip()
        elif line.startswith("CHANNELS:"):
            result["channels"] = line.split(":", 1)[1].strip()
        elif line.startswith("TIMEPOINTS:"):
            result["timepoints"] = line.split(":", 1)[1].strip()
    return result


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_already_processed(path: Path) -> set[int]:
    """Return set of dataset IDs already written to the output CSV."""
    if not path.exists():
        return set()
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        return {int(row["dataset_id"]) for row in reader if row.get("dataset_id")}


In [4]:

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(message)s",
        handlers=[
            logging.StreamHandler(sys.stdout),
            logging.FileHandler(LOG_PATH),
        ],
    )
    log = logging.getLogger(__name__)

    if not API_KEY:
        log.error("No OpenAI API key found. Set OPENAI_API_KEY or set API_KEY in the script.")
        sys.exit(1)

    already_done = load_already_processed(OUTPUT_PATH) if RESUME else set()
    if already_done:
        log.info(f"Resuming: {len(already_done)} datasets already processed.")

    log.info(f"Connecting to OMERO at {SERVER}...")
    conn = connect_omero(SERVER, USERNAME, PASSWORD)
    log.info("Connected.")

    client = openai.OpenAI(api_key=API_KEY)

    try:
        all_datasets = get_all_datasets(conn)
        log.info(f"Found {len(all_datasets)} datasets on the server.")

        datasets_to_process = [
            (ds_id, ds_name)
            for ds_id, ds_name in all_datasets
            if ds_id not in already_done
        ]
        if LIMIT:
            datasets_to_process = datasets_to_process[:LIMIT]
        log.info(f"Will process {len(datasets_to_process)} datasets.")

        # # Open in append mode: new file gets a header; existing file (RESUME=True) does not
        # write_header = not OUTPUT_PATH.exists() or not RESUME
        # fieldnames = [
        #     "dataset_id",
        #     "dataset_name",
        #     "has_log",
        #     "classification",
        #     "reason",
        #     "channels",
        #     "timepoints",
        #     "raw_llm_response",
        # ]
        # csv_file = open(OUTPUT_PATH, "a", newline="")
        # writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        # if write_header:
        #     writer.writeheader()

        # try:
        #     for i, (ds_id, ds_name) in enumerate(datasets_to_process, 1):
        #         log.info(f"[{i}/{len(datasets_to_process)}] Dataset {ds_id}: {ds_name}")

        #         ds_obj = conn.getObject("Dataset", ds_id)
        #         log_text = read_log_annotation(ds_obj)
        #         if log_text is None:
        #             log.warning(f"  No .log/.txt annotation found for dataset {ds_id}.")

        #         result = classify(client, ds_id, ds_name, log_text, model=MODEL)
        #         writer.writerow(result)
        #         csv_file.flush()  # write through after each row so results survive a crash mid-run

        #         log.info(
        #             f"  -> {result['classification']} | {result['reason']} "
        #             f"| channels: {result['channels']} | tps: {result['timepoints']}"
        #         )

        #         # Polite pause to avoid hammering the LLM API
        #         time.sleep(0.2)

        # finally:
        #     csv_file.close()

    finally:
        conn.close()

    # # Summary
    # with open(OUTPUT_PATH, newline="") as f:
    #     rows = list(csv.DictReader(f))
    # yes_count = sum(1 for r in rows if r["classification"] == "YES")
    # log.info(
    #     f"Finished. {yes_count}/{len(rows)} experiments classified as "
    #     "fluorescence time-lapse."
    # )
    # log.info(f"Results saved to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


2026-05-11 13:12:41,971 INFO  [                                __main__] (MainThread) Connecting to OMERO at staffa.bio.ed.ac.uk...
2026-05-11 13:12:41,971 DEBUG [                           omero.gateway] (MainThread) staffa.bio.ed.ac.uk
2026-05-11 13:12:41,972 DEBUG [                           omero.gateway] (MainThread) None
2026-05-11 13:12:41,972 DEBUG [                           omero.gateway] (MainThread) []
2026-05-11 13:12:41,980 DEBUG [                           omero.gateway] (MainThread) Connect attempt, sUuid=None, group=None, self.sUuid=None
2026-05-11 13:12:41,981 DEBUG [                           omero.gateway] (MainThread) Creating Session...
2026-05-11 13:12:42,156 DEBUG [                           omero.gateway] (MainThread) ## Creating proxies
2026-05-11 13:12:42,159 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.client.uuid' to '86c45bf8-26dc-45d3-96ab-8ff59f50ee05'
2026-05-11 13:12:42,161 DEBUG [                     omero.gateway.utils